In [ ]:
import torch

print(torch.cuda.is_available())

!pip install torch_geometric

import torch
from torch import nn
import torch.nn.functional as F
from torch_geometric.datasets import WikipediaNetwork
from torch_geometric.utils import to_undirected, add_self_loops
from torch_geometric.nn import GCNConv, GATConv, GATv2Conv, GENConv, APPNP, MessagePassing
from torch_geometric.utils import softmax
import os
import time
import random
import numpy as np
import torch_geometric.transforms as T

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
hidden_dim = 64
dropout = 0.5
lr = 0.01
weight_decay = 1e-4
patience = 100
max_epochs = 2000
num_runs = 10
base_seed = 42
grad_clip_norm = 1.0
scheduler_patience = 50
scheduler_factor = 0.5
min_lr = 1e-5

class GENGATConv(MessagePassing):
    def __init__(self, in_channels, out_channels, heads=4, concat=True, dropout=0.5):
        super().__init__(aggr='add', node_dim=0)
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.heads = heads
        self.concat = concat
        self.dropout = dropout

        self.lin = nn.Linear(in_channels, heads * out_channels, bias=True)
        self.attn = nn.Parameter(torch.Tensor(heads, 2 * out_channels, 1))
        self.leaky_relu = nn.LeakyReLU(0.2)

        mlp_in = heads * out_channels if concat else out_channels
        self.mlp = nn.Sequential(
            nn.Linear(mlp_in, mlp_in),
            nn.ReLU(),
            nn.Linear(mlp_in, mlp_in)
        )
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.lin.weight, gain=1.414)
        nn.init.zeros_(self.lin.bias)
        nn.init.xavier_uniform_(self.attn, gain=1.414)

    def forward(self, x, edge_index):
        x_transformed = self.lin(x)
        aggr_out = self.propagate(edge_index, x=x_transformed)
        return self.mlp(aggr_out) + x_transformed

    def message(self, x_i, x_j, index):
        x_i = x_i.view(-1, self.heads, self.out_channels)
        x_j = x_j.view(-1, self.heads, self.out_channels)

        attn_input = torch.cat([x_i, x_j], dim=-1)
        e = torch.einsum('ehd,hdo->eh', attn_input, self.attn)
        e = self.leaky_relu(e)

        alpha = softmax(e, index)
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)

        out = alpha.unsqueeze(-1) * x_j
        return out.reshape(-1, self.heads * self.out_channels)

    def update(self, aggr_out):
        if self.concat:
            return aggr_out
        else:
            return aggr_out.reshape(-1, self.heads, self.out_channels).mean(dim=1)

class GCN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

class GAT(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=4, dropout=dropout)
        self.conv2 = GATConv(hidden_channels * 4, out_channels, heads=1, concat=False, dropout=dropout)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

class GATv2(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=4, dropout=dropout)
        self.conv2 = GATv2Conv(hidden_channels * 4, out_channels, heads=1, concat=False, dropout=dropout)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

class GENConvNet(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GENConv(in_channels, hidden_channels, aggr='softmax', t=1.0, learn_t=True, num_layers=2, norm='layer')
        self.conv2 = GENConv(hidden_channels, out_channels, aggr='softmax', t=1.0, learn_t=True, num_layers=2, norm='layer')

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

class GENGATNet(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GENGATConv(in_channels, hidden_channels, heads=4, concat=True, dropout=dropout)
        self.conv2 = GENGATConv(hidden_channels * 4, out_channels, heads=1, concat=False, dropout=dropout)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

class APPNPNet(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, K=10, alpha=0.1):
        super().__init__()
        self.lin1 = nn.Linear(in_channels, hidden_channels)
        self.lin2 = nn.Linear(hidden_channels, out_channels)
        self.prop = APPNP(K=K, alpha=alpha)

    def forward(self, x, edge_index):
        x = F.dropout(x, p=dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=dropout, training=self.training)
        x = self.lin2(x)
        x = self.prop(x, edge_index)
        return x

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

root = os.getcwd()
dataset = WikipediaNetwork(root, name='Chameleon', transform=T.NormalizeFeatures())
#dataset = WikipediaNetwork(root, name='Squirrel', transform=T.NormalizeFeatures())
#dataset = Amazon(root, name='Computers', transform=T.NormalizeFeatures())
#dataset = Amazon(root, name='Photo', transform=T.NormalizeFeatures())
#dataset = Actor(root, transform=T.NormalizeFeatures())
#dataset = Coauthor(root, name='Physics', transform=T.NormalizeFeatures())
original_data = dataset[0]
original_data.edge_index = to_undirected(original_data.edge_index)
original_data.edge_index, _ = add_self_loops(original_data.edge_index, num_nodes=original_data.num_nodes)

num_nodes = original_data.num_nodes
in_channels = dataset.num_features
out_channels = dataset.num_classes

print(f"Dataset: {dataset.name} | Nodes={num_nodes:,}, Edges={original_data.num_edges:,}, Features={in_channels}, Classes={out_channels}")

def train_one_run(data, model):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=scheduler_factor, patience=scheduler_patience,
        min_lr=min_lr
    )

    best_val_acc = test_acc = 0.0
    best_epoch = 0
    counter = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = criterion(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip_norm)
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out = model(data.x, data.edge_index)
            pred = out.argmax(dim=1)
            val_acc = (pred[data.val_mask] == data.y[data.val_mask]).float().mean().item()
            test_acc_cur = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            test_acc = test_acc_cur
            best_epoch = epoch
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                break

        scheduler.step(val_acc)

    return best_val_acc, test_acc, best_epoch

models = [
    ('GCN', GCN),
    ('GAT', GAT),
    ('GATv2', GATv2),
    ('GENConv', GENConvNet),
    ('GENGAT', GENGATNet),
    ('APPNP', APPNPNet),
]

all_results = {}

for model_name, model_class in models:
    print("\n" + "="*80)
    print(f"{model_name}: 10 seeds × 80/10/10 RandomNodeSplit")
    print("="*80)

    val_accs, test_accs, run_times, best_epochs = [], [], [], []
    total_start = time.time()

    for run in range(1, num_runs + 1):
        seed = base_seed + run - 1
        set_seed(seed)
        start = time.time()

        data = original_data.clone()
        split_transform = T.RandomNodeSplit(split='train_rest', num_val=0.10, num_test=0.10)
        data = split_transform(data)
        data = data.to(device)

        model = model_class(in_channels, hidden_dim, out_channels).to(device)
        best_val, best_test, best_epoch = train_one_run(data, model)
        elapsed = time.time() - start

        val_accs.append(best_val)
        test_accs.append(best_test)
        run_times.append(elapsed)
        best_epochs.append(best_epoch)

        print(f"Run {run:2d} | Seed {seed:2d} → Epoch {best_epoch:4d} | Val {best_val*100:6.2f}% | Test {best_test*100:6.2f}% | {elapsed:5.2f}s")

    mean_test = np.mean(test_accs)
    std_test = np.std(test_accs)

    all_results[model_name] = {
        'mean_test': mean_test,
        'std_test': std_test,
        'best_test': max(test_accs),
        'mean_epochs': np.mean(best_epochs),
        'mean_time': np.mean(run_times),
        'total_time': time.time() - total_start
    }

    print("\n" + "="*80)
    print(f"RESULTS: {model_name}")
    print("="*80)
    print(f"Test Acc.:      {mean_test*100:.2f}±{std_test*100:.2f}%")
    print(f"Best Test Acc.: {max(test_accs)*100:.2f}%")
    print(f"Mean Epochs:    {np.mean(best_epochs):.1f}")
    print(f"Mean Runtime:   {np.mean(run_times):.2f}s")
    print(f"Total Time:     {time.time() - total_start:.1f}s")
    print("="*80)

print("\n" + "="*80)
print("FINAL COMPARISON: All Models")
print("="*80)
print(f"{'Model':<15} {'Test Acc (%)':<20} {'Best (%)':<12} {'Epochs':<10} {'Time/run (s)':<15} {'Total (s)':<12}")
print("-"*80)
for model_name in ['GCN', 'GAT', 'GATv2', 'GENConv', 'GENGAT', 'APPNP']:
    r = all_results[model_name]
    print(f"{model_name:<15} {r['mean_test']*100:.2f}±{r['std_test']*100:.2f}{'':<8} {r['best_test']*100:.2f}{'':<6} {r['mean_epochs']:<10.1f} {r['mean_time']:<15.2f} {r['total_time']:<12.1f}")
print("="*80)